In [1]:
!pip install -q sahi ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 30.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 10.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [2]:
# ─────────────────────────────────────────────
# CELL 1 — Imports + Path Config
# ─────────────────────────────────────────────
import subprocess, cv2, numpy as np, pandas as pd
from pathlib import Path
from collections import defaultdict, deque

# ── Paths ──────────────────────────────────────
INPUT_DIR  = Path("/kaggle/input/datasets/vaibhavsonkar/cricketballtracking")
CFR_DIR    = Path("/kaggle/working/cfr")
OUTPUT_DIR = Path("/kaggle/working/output")
WEIGHTS    = Path("/kaggle/input/datasets/vaibhavsonkar/cricketballtracking/best.pt")

for d in [CFR_DIR, OUTPUT_DIR / "csv", OUTPUT_DIR / "video"]:
    d.mkdir(parents=True, exist_ok=True)

# ── Detection / tracking hyper-params ──────────
# ── Detection / tracking hyper-params ──────────
CONF_THRESH    = 0.35   # updated from diagnostic
CONF_REINIT    = 0.50   # stricter gate for INIT and LOST states
SLICE_SIZE     = 640
OVERLAP_RATIO  = 0.2
TRAIL_LENGTH   = 15
INTERP_MAX_GAP = 5

# ── Ball filter constants (calibrated for 2560px frame) ──
MIN_BALL_SIZE  = 11     # px — p5 from diagnostic
MAX_BALL_SIZE  = 35     # px — kills static FPs (71px blob)
MIN_ASPECT     = 0.4    # width/height ratio gate
MAX_ASPECT     = 2.5

# ── Exclusion zones (fraction of frame) ──────────
EXCL_ZONES = [
    (0.00, 0.85, 1.00, 1.00),  # bottom 15% — scoreboard
    (0.75, 0.00, 1.00, 0.18),  # top-right   — logo / live badge
]

# ── Displacement gates (px, calibrated for 2560px @ 60fps) ──
DISP_TRACKING  = 60    # hard gate when TRACKING
DISP_REINIT    = 200   # loose gate when INIT or LOST
MIN_GATE       = 40    # floor so gate never collapses on slow ball

print("✓ Cell 1 ready")
print(f"  WEIGHTS exists: {WEIGHTS.exists()}")

✓ Cell 1 ready
  WEIGHTS exists: True


In [4]:
# ─────────────────────────────────────────────
# CELL 2 — CFR Conversion  (skips existing files)
# ─────────────────────────────────────────────
videos = sorted(INPUT_DIR.glob("*.mov")) + sorted(INPUT_DIR.glob("*.mp4"))

if not videos:
    raise FileNotFoundError(f"No video files found in {INPUT_DIR}")

passed, failed, skipped = [], [], []

for vid in videos:
    if vid.name == "1.mp4":
        out = CFR_DIR / "1.mp4"
        if out.exists():
            print(f"  skipped 1.mp4 (already exists)")
            skipped.append(out)
            continue
        subprocess.run(["cp", str(vid), str(out)], check=True)
        print(f"  copied  1.mp4 as-is (already CFR)")
        passed.append(out)
        continue

    out = CFR_DIR / f"{vid.stem}_cfr.mp4"
    if out.exists():
        print(f"  skipped {vid.name:20s} → {out.name} (already exists)")
        skipped.append(out)
        continue

    cmd = [
        "ffmpeg", "-y",
        "-i",      str(vid),
        "-vf",     "fps=60",
        "-c:v",    "libx264",
        "-preset", "fast",
        "-crf",    "18",
        "-an",
        str(out),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"  ✓ {vid.name:20s} → {out.name}")
        passed.append(out)
    else:
        print(f"  ✗ {vid.name:20s} FAILED: {result.stderr[-200:].strip()}")
        failed.append(vid.name)

print(f"\n✓ Cell 2 done — {len(passed)} converted, {len(skipped)} skipped, {len(failed)} failed")
if failed:
    print(f"  Failed: {failed}")

  skipped 10.mov               → 10_cfr.mp4 (already exists)
  skipped 11.mov               → 11_cfr.mp4 (already exists)
  skipped 12.mov               → 12_cfr.mp4 (already exists)
  skipped 13.mov               → 13_cfr.mp4 (already exists)
  skipped 14.mov               → 14_cfr.mp4 (already exists)
  skipped 15.mov               → 15_cfr.mp4 (already exists)
  skipped 2.mov                → 2_cfr.mp4 (already exists)
  skipped 3.mov                → 3_cfr.mp4 (already exists)
  skipped 4.mov                → 4_cfr.mp4 (already exists)
  skipped 5.mov                → 5_cfr.mp4 (already exists)
  skipped 6.mov                → 6_cfr.mp4 (already exists)
  skipped 7.mov                → 7_cfr.mp4 (already exists)
  skipped 8.mov                → 8_cfr.mp4 (already exists)
  skipped 9.mov                → 9_cfr.mp4 (already exists)
  skipped 1.mp4 (already exists)

✓ Cell 2 done — 0 converted, 15 skipped, 0 failed


In [5]:
# CELL 3 — Model Loading

import torch
from sahi import AutoDetectionModel
 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU    : {torch.cuda.get_device_name(0)}")
 
if not WEIGHTS.exists():
    raise FileNotFoundError(
        f"Weights not found at {WEIGHTS}\n"
        "Ensure best.pt is added to your Kaggle dataset and INPUT_DIR is correct."
    )
 
detection_model = AutoDetectionModel.from_pretrained(
    model_type           = "ultralytics",
    model_path           = str(WEIGHTS),
    confidence_threshold = CONF_THRESH,
    device               = DEVICE,
)
 
print(f"✓ Cell 3 done — model loaded ({WEIGHTS.name})")
print(f"  Confidence threshold : {CONF_THRESH}")
print(f"  Slice size / overlap : {SLICE_SIZE}px / {OVERLAP_RATIO}")

  Device : cuda
  GPU    : Tesla T4
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ Cell 3 done — model loaded (best.pt)
  Confidence threshold : 0.35
  Slice size / overlap : 640px / 0.2


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Frame Detection + Static Prescan
# ─────────────────────────────────────────────────────────────────────────────
# Two functions only:
#   prescan_blacklist(cap, model)            — run once per video before main loop
#   detect_frame(frame_bgr, blacklist,
#                excl_px, model, min_conf)   — run on every frame, returns candidates list
# ─────────────────────────────────────────────────────────────────────────────

from sahi.predict import get_sliced_prediction

PRESCAN_FRAMES = [0, 5, 10]   # frames to scan for permanent static FPs
PRESCAN_HITS   = 2            # fires in ≥ this many → blacklisted
STATIC_GRID    = 30           # px — snap to grid for static FP grouping


def prescan_blacklist(cap: cv2.VideoCapture, model) -> set:
    """
    Sample PRESCAN_FRAMES from an open VideoCapture.
    Return a set of (gx, gy) grid cells that fired in ≥ PRESCAN_HITS frames.
    These are permanent static objects — logo, scoreboard, ad banner.
    Cap position is restored to frame 0 after.
    """
    total       = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cell_counts = defaultdict(int)

    for fi in PRESCAN_FRAMES:
        if fi >= total:
            continue
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame_bgr = cap.read()
        if not ret:
            continue

        result = get_sliced_prediction(
            cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB),
            model,                                        # ← thread-local model
            slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
            overlap_height_ratio=OVERLAP_RATIO,
            overlap_width_ratio=OVERLAP_RATIO,
            verbose=0,
        )
        for pred in result.object_prediction_list:
            cx = (pred.bbox.minx + pred.bbox.maxx) / 2.0
            cy = (pred.bbox.miny + pred.bbox.maxy) / 2.0
            cell_counts[(int(cx // STATIC_GRID), int(cy // STATIC_GRID))] += 1

    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    blacklist = {cell for cell, n in cell_counts.items() if n >= PRESCAN_HITS}
    print(f"    prescan: {len(blacklist)} static cells blacklisted")
    return blacklist


def _build_excl_px(frame_w: int, frame_h: int) -> list:
    """Convert fractional EXCL_ZONES to absolute pixel rectangles."""
    return [
        (int(x0*frame_w), int(y0*frame_h), int(x1*frame_w), int(y1*frame_h))
        for x0, y0, x1, y1 in EXCL_ZONES
    ]


def detect_frame(frame_bgr:  np.ndarray,
                 blacklist:  set,
                 excl_px:    list,
                 model,                                   # ← thread-local model
                 min_conf:   float = None) -> list:
    """
    Run SAHI on one frame and return filtered candidates.

    Filters applied in order (cheapest first):
      1. Exclusion zones      — hard pixel regions (scoreboard, logo)
      2. Prescan blacklist    — static grid cells from prescan
      3. Confidence           — uses min_conf arg (caller sets per state)
      4. Size                 — MIN_BALL_SIZE to MAX_BALL_SIZE
      5. Aspect ratio         — MIN_ASPECT to MAX_ASPECT

    Returns
    -------
    list of {"cx": float, "cy": float, "conf": float}
    sorted by confidence descending. Empty list = no ball found.
    """
    if min_conf is None:
        min_conf = CONF_THRESH

    result = get_sliced_prediction(
        cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB),
        model,                                            # ← thread-local model
        slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
        overlap_height_ratio=OVERLAP_RATIO,
        overlap_width_ratio=OVERLAP_RATIO,
        verbose=0,
    )

    candidates = []
    for pred in result.object_prediction_list:
        cx   = (pred.bbox.minx + pred.bbox.maxx) / 2.0
        cy   = (pred.bbox.miny + pred.bbox.maxy) / 2.0
        bw   = pred.bbox.maxx - pred.bbox.minx
        bh   = pred.bbox.maxy - pred.bbox.miny
        conf = pred.score.value

        # 1. Exclusion zones
        if any(x0 <= cx <= x1 and y0 <= cy <= y1
               for x0, y0, x1, y1 in excl_px):
            continue

        # 2. Prescan blacklist
        if (int(cx // STATIC_GRID), int(cy // STATIC_GRID)) in blacklist:
            continue

        # 3. Confidence
        if conf < min_conf:
            continue

        # 4. Size
        if not (MIN_BALL_SIZE <= max(bw, bh) <= MAX_BALL_SIZE):
            continue

        # 5. Aspect ratio
        ar = bw / bh if bh > 0 else 99.0
        if not (MIN_ASPECT <= ar <= MAX_ASPECT):
            continue

        candidates.append({"cx": cx, "cy": cy, "conf": conf})

    return sorted(candidates, key=lambda c: c["conf"], reverse=True)


print("✓ Cell 4 ready")
print(f"  Filters : excl zones → blacklist → conf → size({MIN_BALL_SIZE}–{MAX_BALL_SIZE}px) → aspect")

✓ Cell 4 ready
  Filters : excl zones → blacklist → conf → size(11–35px) → aspect


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — PhysicsTracker
# ─────────────────────────────────────────────────────────────────────────────
# Single class. Three states: INIT → TRACKING → LOST
#
# State       conf gate      disp gate         Kalman
# ─────────────────────────────────────────────────────
# INIT        CONF_REINIT    none               off
# TRACKING    CONF_THRESH    dynamic (speed)    on — predict + correct
# LOST        CONF_REINIT    DISP_REINIT        on — predict only
#
# Kalman state: [cx, cy, vx, vy]
# Physics additions to vanilla CV model:
#   vy += GRAVITY  each predict step  (ball falls in pixel space)
#   vx *= DRAG     each predict step  (air resistance)
#   vy *= DRAG     each predict step  (air resistance — same as vx)
#
# Dynamic gate (TRACKING only):
#   gate = max(current_speed * GATE_FACTOR, MIN_GATE)
#   Ball moving fast → bigger window. Stationary → tight window.
#   This is the fix for both the zigzag FP problem AND missing a fast ball.
# ─────────────────────────────────────────────────────────────────────────────

GRAVITY      = 1.5    # px/frame added to vy each step
DRAG         = 0.98   # vx and vy multiplied each step
GATE_FACTOR  = 2.0    # gate = speed × this
MAX_MISS     = 3      # frames before TRACKING → LOST
                      # tracker Kalman-fills for MAX_MISS-1 frames,
                      # then goes LOST on the MAX_MISS-th consecutive miss

# tracker states
INIT     = "INIT"
TRACKING = "TRACKING"
LOST     = "LOST"


def _make_kalman() -> cv2.KalmanFilter:
    kf = cv2.KalmanFilter(4, 2)
    kf.transitionMatrix = np.array([
        [1, 0, 1, 0],
        [0, 1, 0, 1],
        [0, 0, DRAG, 0],   # vx decays by drag each step
        [0, 0, 0, DRAG],   # vy decays by drag each step  ← FIX Bug 2
    ], dtype=np.float32)
    kf.measurementMatrix   = np.array([[1,0,0,0],[0,1,0,0]], dtype=np.float32)
    kf.processNoiseCov     = np.eye(4, dtype=np.float32) * 1e-2
    kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 5.0
    kf.errorCovPost        = np.eye(4, dtype=np.float32) * 1e4
    return kf


class PhysicsTracker:

    def __init__(self):
        self.state       = INIT
        self.kf          = _make_kalman()
        self.miss_count  = 0
        self.speed       = 0.0   # px/frame, updated each detection

    # ── public ────────────────────────────────────────────────────────────────

    def update(self, candidates: list) -> dict:
        """
        Advance tracker by one frame.

        Parameters
        ----------
        candidates : output of detect_frame() — already confidence-filtered
                     at the CORRECT threshold for current state.
                     List of {"cx", "cy", "conf"}, sorted best-first.

        Returns
        -------
        dict with keys: cx, cy, visible (0/1), state, source
        cx/cy = None when visible=0
        """
        if self.state == INIT:
            return self._update_init(candidates)
        elif self.state == TRACKING:
            return self._update_tracking(candidates)
        else:
            return self._update_lost(candidates)

    @property
    def conf_gate(self) -> float:
        """Confidence threshold the CALLER should use for detect_frame."""
        return CONF_THRESH if self.state == TRACKING else CONF_REINIT

    # ── state handlers ─────────────────────────────────────────────────────────

    def _update_init(self, candidates: list) -> dict:
        if not candidates:
            return self._null(INIT, "no_det")

        best = candidates[0]
        self._kalman_init(best["cx"], best["cy"])
        self.state      = TRACKING
        self.miss_count = 0
        self.speed      = 0.0
        return self._hit(best["cx"], best["cy"], TRACKING, "detection")

    def _update_tracking(self, candidates: list) -> dict:
        self.kf.predict()
        # apply gravity to the freshly predicted vy before reading position  ← FIX Bug 1
        self.kf.statePre[3] += GRAVITY
        px = float(self.kf.statePre[0].item())
        py = float(self.kf.statePre[1].item())

        # dynamic gate based on current ball speed
        gate = max(self.speed * GATE_FACTOR, MIN_GATE)

        # pick best candidate within gate
        best = self._best_in_gate(candidates, px, py, gate)

        if best:
            self.kf.correct(np.array(
                [[np.float32(best["cx"])], [np.float32(best["cy"])]]
            ))
            vx = float(self.kf.statePost[2].item())
            vy = float(self.kf.statePost[3].item())
            self.speed      = np.hypot(vx, vy)
            self.miss_count = 0
            return self._hit(best["cx"], best["cy"], TRACKING, "detection")

        # no detection in gate — use Kalman prediction
        self.miss_count += 1
        if self.miss_count < MAX_MISS:                   # ← FIX Bug 3  (< not <=)
            return self._hit(px, py, TRACKING, "kalman_predict")

        # too many misses — go LOST
        self.state = LOST
        return self._null(LOST, "lost")

    def _update_lost(self, candidates: list) -> dict:
        # still run predict to keep Kalman state warm
        self.kf.predict()
        self.kf.statePre[3] += GRAVITY                   # ← FIX Bug 1 (same fix here)
        px = float(self.kf.statePre[0].item())
        py = float(self.kf.statePre[1].item())

        # strict gate for reacquisition
        best = self._best_in_gate(candidates, px, py, DISP_REINIT)

        if best:
            # re-initialise cleanly from this detection
            self._kalman_init(best["cx"], best["cy"])
            self.state      = TRACKING
            self.miss_count = 0
            self.speed      = 0.0
            return self._hit(best["cx"], best["cy"], TRACKING, "detection")

        return self._null(LOST, "lost")

    # ── helpers ────────────────────────────────────────────────────────────────

    def _kalman_init(self, cx: float, cy: float):
        self.kf.statePost    = np.array([[cx],[cy],[0.],[0.]], dtype=np.float32)
        self.kf.statePre     = np.array([[cx],[cy],[0.],[0.]], dtype=np.float32)
        self.kf.errorCovPost = np.eye(4, dtype=np.float32) * 1e4

    def _best_in_gate(self, candidates: list,
                      px: float, py: float, gate: float):
        """Return highest-conf candidate within gate, or None."""
        for c in candidates:   # already sorted best-first
            if np.hypot(c["cx"] - px, c["cy"] - py) <= gate:
                return c
        return None

    @staticmethod
    def _hit(cx, cy, state, source) -> dict:
        return {"cx": cx, "cy": cy, "visible": 1,
                "state": state, "source": source}

    @staticmethod
    def _null(state, source) -> dict:
        return {"cx": None, "cy": None, "visible": 0,
                "state": state, "source": source}


print("✓ Cell 5 ready — PhysicsTracker defined")
print(f"  States  : INIT → TRACKING → LOST")
print(f"  Gravity : {GRAVITY}px/frame on vy (applied to statePre after predict)")
print(f"  Drag    : vx and vy × {DRAG} per frame")
print(f"  Gate    : max(speed × {GATE_FACTOR}, {MIN_GATE}px)  dynamic")
print(f"  MaxMiss : {MAX_MISS} frames — Kalman fills {MAX_MISS-1}, LOST on {MAX_MISS}th miss")

✓ Cell 5 ready — PhysicsTracker defined
  States  : INIT → TRACKING → LOST
  Gravity : 1.5px/frame on vy (applied to statePre after predict)
  Drag    : vx and vy × 0.98 per frame
  Gate    : max(speed × 2.0, 40px)  dynamic
  MaxMiss : 3 frames — Kalman fills 2, LOST on 3th miss


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Gap Interpolation + CSV Export
# ─────────────────────────────────────────────────────────────────────────────
# Input  : tracker output — list of dicts from PhysicsTracker.update()
# Output : cleaned list + CSV written to disk
#
# Three tiers of output (in order of confidence):
#   1. source="detection"      → visible=1, real coordinates
#   2. source="kalman_predict" → visible=1, Kalman-filled (Cell 5 already did this)
#   3. gap ≤ INTERP_MAX_GAP    → visible=1, linear interpolation between anchors
#   4. everything else         → visible=0, x=-1, y=-1  (spec requirement)
#
# This cell only touches tier 3 and 4.
# Tiers 1 and 2 come out of Cell 5 untouched.
# ─────────────────────────────────────────────────────────────────────────────

MAX_INTERP_JUMP = 150   # px — if anchors are further apart than this,
                        # they belong to different events (end of one delivery,
                        # start of next). Do NOT interpolate between them.

def interpolate_and_export(tracked: list, csv_path: str) -> list:
    """
    Fill remaining lost-frame gaps via linear interpolation, write CSV.

    Parameters
    ----------
    tracked   : list of dicts from PhysicsTracker — one per frame
    csv_path  : destination path for CSV file

    Returns
    -------
    list of {"frame", "x", "y", "visible"} — one per frame
    """
    n      = len(tracked)
    filled = []

    # ── Pass 1: flatten tracker output to (frame, cx, cy, visible, source) ──
    for r in tracked:
        if r["visible"] == 1:
            filled.append([r["frame"], r["cx"], r["cy"], 1, r["source"]])
        else:
            filled.append([r["frame"], None, None, 0, "lost"])

    # ── Pass 2: find lost runs and decide interpolate vs absent ──────────────
    i = 0
    while i < n:
        if filled[i][1] is not None:   # known position — skip
            i += 1
            continue

        # found start of a lost run
        gap_start = i
        while i < n and filled[i][1] is None:
            i += 1
        gap_end = i   # first known frame after gap (exclusive)
        gap_len = gap_end - gap_start

        # anchors
        left  = filled[gap_start - 1] if gap_start > 0 else None
        right = filled[gap_end]        if gap_end   < n else None

        anchor_dist = (
            np.hypot(right[1] - left[1], right[2] - left[2])
            if (left and left[1] is not None and right and right[1] is not None)
            else float("inf")
        )

        can_interp = (
            left  is not None and left[1]  is not None and
            right is not None and right[1] is not None and
            gap_len     <= INTERP_MAX_GAP and
            anchor_dist <= MAX_INTERP_JUMP   # same ball, same delivery
        )

        for j in range(gap_start, gap_end):
            if can_interp:
                t           = (j - gap_start + 1) / (gap_len + 1)
                filled[j][1] = left[1] + t * (right[1] - left[1])
                filled[j][2] = left[2] + t * (right[2] - left[2])
                filled[j][3] = 1
                filled[j][4] = "interpolated"
            else:
                filled[j][1] = -1.0
                filled[j][2] = -1.0
                filled[j][3] = 0
                filled[j][4] = "absent"

    # ── Pass 3: build rows and write CSV ─────────────────────────────────────
    rows = []
    for (fi, cx, cy, vis, src) in filled:
        if vis == 1:
            rows.append({"frame": fi, "x": round(float(cx), 1),
                 "y": round(float(cy), 1), "visible": 1, "source": src})
        else:
            rows.append({"frame": fi, "x": -1, "y": -1, "visible": 0})

    Path(csv_path).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows, columns=["frame", "x", "y", "visible"]) \
      .to_csv(csv_path, index=False)

    # ── Summary ───────────────────────────────────────────────────────────────
    n_vis    = sum(1 for r in rows if r["visible"] == 1)
    n_absent = len(rows) - n_vis
    src_counts = defaultdict(int)
    for f in filled:
        src_counts[f[4]] += 1

    print(f"    {Path(csv_path).name}")
    print(f"    {len(rows)} frames  |  visible={n_vis}  absent={n_absent}")
    print(f"    sources: det={src_counts['detection']}  "
          f"kalman={src_counts['kalman_predict']}  "
          f"interp={src_counts['interpolated']}  "
          f"absent={src_counts['absent']}")

    return rows


print("✓ Cell 6 ready — interpolate_and_export() defined")
print(f"  Interpolation window : ≤ {INTERP_MAX_GAP} frames")
print(f"  Beyond window        : x=-1, y=-1, visible=0")

✓ Cell 6 ready — interpolate_and_export() defined
  Interpolation window : ≤ 5 frames
  Beyond window        : x=-1, y=-1, visible=0


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Overlay Video Writer
# ─────────────────────────────────────────────────────────────────────────────
# Reads CFR video frame by frame, draws:
#   - Fading trail        : last TRAIL_LENGTH visible positions
#   - Centroid dot        : current ball position
#   - State label         : INIT / TRACKING / LOST  (top-left)
#   - Frame counter       : bottom-left
#
# Colour encodes source:
#   Yellow  (#FFD700)  = real detection
#   Cyan    (#00FFFF)  = kalman_predict or interpolated
#   Nothing drawn      = visible=0  (ball absent — no dot, no trail addition)
#
# Trail only grows when visible=1.
# visible=0 frames: trail is drawn from history but NOT extended.
# This means the trail naturally fades out and disappears during non-play.
# ─────────────────────────────────────────────────────────────────────────────

COL_DETECT  = (0,   215, 255)   # BGR yellow
COL_KALMAN  = (255, 255,   0)   # BGR cyan
COL_ABSENT  = (0,    0,  180)   # BGR red  — "BALL NOT VISIBLE" text
COL_STATE   = (200, 200, 200)   # BGR light grey — state label
FONT        = cv2.FONT_HERSHEY_SIMPLEX
DOT_RADIUS  = 7


def write_overlay(cfr_path: str,
                  final_rows: list, tracked: list,
                  out_path: str) -> None:
    """
    Parameters
    ----------
    cfr_path   : path to CFR-converted input video
    final_rows : output of interpolate_and_export() —
                 list of {"frame", "x", "y", "visible"}
    out_path   : destination MP4 path
    """
    cap    = cv2.VideoCapture(cfr_path)
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(
        out_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps, (width, height)
    )

    # O(1) lookup: frame_idx → row
    lookup = {r["frame"]: r for r in final_rows}
    state_lookup = {r["frame"]: r.get("state", "") for r in tracked}

    # trail stores (x, y, source) of recent visible positions
    trail: deque = deque(maxlen=TRAIL_LENGTH)

    for frame_idx in range(total):
        ret, frame = cap.read()
        if not ret:
            break

        rec = lookup.get(frame_idx)

        # ── extend trail only on visible frames ───────────────────────────
        if rec and rec["visible"] == 1:
            # source comes from tracked list; fall back to "detection"
            src = rec.get("source", "detection")
            trail.append((rec["x"], rec["y"], src))

        # ── draw fading trail ─────────────────────────────────────────────
        pts = list(trail)
        for k in range(1, len(pts)):
            alpha = 0.25 + 0.75 * (k / len(pts))   # 25% → 100%
            col   = COL_DETECT if pts[k][2] == "detection" else COL_KALMAN
            ov    = frame.copy()
            cv2.line(ov,
                     (int(pts[k-1][0]), int(pts[k-1][1])),
                     (int(pts[k][0]),   int(pts[k][1])),
                     col, 2, cv2.LINE_AA)
            cv2.addWeighted(ov, alpha, frame, 1 - alpha, 0, frame)

        # ── draw centroid dot (visible frames only) ───────────────────────
        if rec and rec["visible"] == 1:
            cx_i = int(rec["x"])
            cy_i = int(rec["y"])
            col  = COL_DETECT if rec.get("source","detection") == "detection" \
                   else COL_KALMAN
            # black ring for contrast on any background
            cv2.circle(frame, (cx_i, cy_i), DOT_RADIUS + 2,
                       (0, 0, 0), 2, cv2.LINE_AA)
            cv2.circle(frame, (cx_i, cy_i), DOT_RADIUS,
                       col, -1, cv2.LINE_AA)

        # ── state label (top-left) ────────────────────────────────────────
        state_label = state_lookup.get(frame_idx, "")
        if state_label == LOST:
            cv2.putText(frame, "BALL NOT VISIBLE",
                        (20, 44), FONT, 1.0, COL_ABSENT, 2, cv2.LINE_AA)
        elif state_label:
            cv2.putText(frame, state_label,
                        (20, 44), FONT, 1.0, COL_STATE, 2, cv2.LINE_AA)

        # ── frame counter (bottom-left) ───────────────────────────────────
        cv2.putText(frame, f"f{frame_idx}",
                    (16, height - 16), FONT, 0.5,
                    (160, 160, 160), 1, cv2.LINE_AA)

        writer.write(frame)

    cap.release()
    writer.release()

    # ── re-encode to H264 for smaller file + browser playback ────────────
    h264 = out_path.replace(".mp4", "_h264.mp4")
    r    = subprocess.run(
        ["ffmpeg", "-y", "-i", out_path,
         "-c:v", "libx264", "-preset", "fast", "-crf", "23", "-an", h264],
        capture_output=True
    )
    if r.returncode == 0:
        Path(out_path).unlink()
        Path(h264).rename(out_path)
        codec = "H264"
    else:
        codec = "mp4v"

    mb = Path(out_path).stat().st_size / 1e6
    print(f"    {Path(out_path).name}  [{codec}  {mb:.1f} MB]")


print("✓ Cell 7 ready — write_overlay() defined")
print(f"  Trail : {TRAIL_LENGTH} frames, fades 25%→100%")
print(f"  Yellow = detection  |  Cyan = kalman/interpolated  |  No dot = absent")

✓ Cell 7 ready — write_overlay() defined
  Trail : 15 frames, fades 25%→100%
  Yellow = detection  |  Cyan = kalman/interpolated  |  No dot = absent


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Main Loop  (Dual GPU, ThreadPoolExecutor)
# ─────────────────────────────────────────────────────────────────────────────

import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm


def process_one_video(cfr_path: Path, device: str) -> dict:

    stem     = cfr_path.stem
    csv_path = str(OUTPUT_DIR / "csv"   / f"{stem}.csv")
    vid_path = str(OUTPUT_DIR / "video" / f"{stem}_overlay.mp4")

    try:
        # ── build model on correct device for this thread ─────────────────────
        # No set_device call — from_pretrained handles placement via device=
        from sahi import AutoDetectionModel
        model = AutoDetectionModel.from_pretrained(
            model_type           = "ultralytics",
            model_path           = str(WEIGHTS),
            confidence_threshold = CONF_THRESH,
            device               = device,
        )

        # ── open video ────────────────────────────────────────────────────────
        import cv2
        cap   = cv2.VideoCapture(str(cfr_path))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fw    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        fh    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # ── exclusion zones + prescan blacklist ───────────────────────────────
        excl_px   = _build_excl_px(fw, fh)
        blacklist = prescan_blacklist(cap, model)

        # ── frame loop ────────────────────────────────────────────────────────
        tracker = PhysicsTracker()
        tracked = []

        for fi in range(total):
            ret, frame_bgr = cap.read()
            if not ret:
                tracked.append({"frame": fi, "cx": None, "cy": None,
                                 "visible": 0, "source": "lost", "state": tracker.state})
                continue

            cands  = detect_frame(frame_bgr, blacklist, excl_px,
                                  model, min_conf=tracker.conf_gate)
            result = tracker.update(cands)
            result["frame"] = fi
            tracked.append(result)

        cap.release()

        # ── interpolation + CSV ───────────────────────────────────────────────
        final = interpolate_and_export(tracked, csv_path)

        # ── overlay video ─────────────────────────────────────────────────────
        write_overlay(str(cfr_path), final, tracked, vid_path)

        n_vis = sum(1 for r in final if r["visible"] == 1)
        print(f"  ✓ {cfr_path.name} on {device} — {n_vis}/{len(final)} visible", flush=True)

        return {"video": cfr_path.name, "device": device,
                "frames": len(final), "visible": n_vis, "status": "ok"}

    except Exception:
        import traceback
        return {"video": cfr_path.name, "device": device,
                "frames": 0, "visible": 0,
                "status": f"ERROR: {traceback.format_exc()[-300:]}"}


# ── GPU detection ──────────────────────────────────────────────────────────────
n_gpus = torch.cuda.device_count()
if n_gpus >= 2:
    devices   = ["cuda:0", "cuda:1"]; n_workers = 2
    print(f"✓ Dual-GPU: {torch.cuda.get_device_name(0)} + {torch.cuda.get_device_name(1)}")
elif n_gpus == 1:
    devices   = ["cuda:0"]; n_workers = 1
    print(f"✓ Single-GPU: {torch.cuda.get_device_name(0)}")
else:
    devices   = ["cpu"]; n_workers = 1
    print("⚠ CPU mode")

# ── gather + assign ────────────────────────────────────────────────────────────
cfr_videos = sorted(CFR_DIR.glob("*.mp4"))
if not cfr_videos:
    raise FileNotFoundError(f"No CFR videos in {CFR_DIR}")

assignments = [(v, devices[i % len(devices)]) for i, v in enumerate(cfr_videos)]
print(f"  Videos: {len(cfr_videos)}  Workers: {n_workers}")
for dev in devices:
    print(f"  {dev} → {[v.name for v, d in assignments if d == dev]}")

# ── run ────────────────────────────────────────────────────────────────────────
summaries = []
pbar = tqdm(total=len(assignments), desc="Processing", unit="video")

with ThreadPoolExecutor(max_workers=n_workers) as executor:
    futures = {executor.submit(process_one_video, v, d): v.name
               for v, d in assignments}
    for fut in as_completed(futures):
        res = fut.result()
        summaries.append(res)
        pbar.set_postfix_str(f"{'✓' if res['status']=='ok' else '✗'} {res['video']}")
        pbar.update(1)

pbar.close()

# ── summary ────────────────────────────────────────────────────────────────────
print("\n" + "─"*65)
print(f"{'Video':<22} {'Device':<10} {'Frames':>7} {'Visible':>8} {'%':>5}")
print("─"*65)
for s in sorted(summaries, key=lambda x: x["video"]):
    pct = f"{100*s['visible']/s['frames']:.0f}%" if s["frames"] else "—"
    print(f"{s['video']:<22} {s['device']:<10} {s['frames']:>7} "
          f"{s['visible']:>8} {pct:>5}  {'✓' if s['status']=='ok' else s['status'][:40]}")
print("─"*65)

errors = [s for s in summaries if s["status"] != "ok"]
if errors:
    print(f"\n⚠ {len(errors)} failed")
    for e in errors: print(f"  {e['video']}: {e['status'][:120]}")
else:
    print(f"\n✓ All {len(summaries)} done")
    print(f"  CSVs   → {OUTPUT_DIR/'csv'}")
    print(f"  Videos → {OUTPUT_DIR/'video'}")

✓ Dual-GPU: Tesla T4 + Tesla T4
  Videos: 15  Workers: 2
  cuda:0 → ['1.mp4', '11_cfr.mp4', '13_cfr.mp4', '15_cfr.mp4', '3_cfr.mp4', '5_cfr.mp4', '7_cfr.mp4', '9_cfr.mp4']
  cuda:1 → ['10_cfr.mp4', '12_cfr.mp4', '14_cfr.mp4', '2_cfr.mp4', '4_cfr.mp4', '6_cfr.mp4', '8_cfr.mp4']


Processing:   0%|          | 0/15 [00:00<?, ?video/s]

    prescan: 1 static cells blacklisted
    prescan: 3 static cells blacklisted
    1.csv
    31 frames  |  visible=13  absent=18
    sources: det=6  kalman=4  interp=3  absent=18
    1_overlay.mp4  [H264  1.0 MB]
  ✓ 1.mp4 on cuda:0 — 13/31 visible
    prescan: 4 static cells blacklisted
    11_cfr.csv
    237 frames  |  visible=136  absent=101
    sources: det=104  kalman=22  interp=10  absent=101
    10_cfr.csv
    282 frames  |  visible=168  absent=114
    sources: det=143  kalman=21  interp=4  absent=114
    11_cfr_overlay.mp4  [H264  3.0 MB]
  ✓ 11_cfr.mp4 on cuda:0 — 136/237 visible
    prescan: 6 static cells blacklisted
    10_cfr_overlay.mp4  [H264  2.2 MB]
  ✓ 10_cfr.mp4 on cuda:1 — 168/282 visible
    prescan: 2 static cells blacklisted
    12_cfr.csv
    176 frames  |  visible=4  absent=172
    sources: det=2  kalman=2  interp=0  absent=172
    12_cfr_overlay.mp4  [H264  1.8 MB]
  ✓ 12_cfr.mp4 on cuda:1 — 4/176 visible
    prescan: 5 static cells blacklisted
    13_cfr.csv